## For positive and negative neuron controls within 80K NGN2 derived neurons
- 'C_negative_neuron_NP', 'C_positive_neuron_NP', 'C_positive_neuron_MK',  'C_negative_neuron_MK', 'C_positive_heart_MK', 'C_negative_heart_MK', 'C_positive_neuron_CD'

In [6]:
from importlib import reload
import pandas as pd
import sys
import os
import yaml

sys.path.append('../helpful_functions')
import helpful_functions as hf
reload(hf)

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/config_file.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

In [ ]:
# helpful functions

# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'

def get_category(header):
    """Get the category of the headers set all to element if not scarmbled
    """

    label = hf.get_label(header)

    if 'scramble' in header:
        # Check if header is scrambled
        # Info: scrambled is in C_negative_neuron_NP and scramble MK
        # Note: Other cases are not checked with this function
        return 'scrambled'
    else:
        return 'element'


def get_reference_genome(row):
    """If col_category is synthetic or scrambled or in dnase_control_groups, set ref to GRCh37 else GRCh38"""
    row[col_ref] = 'GRCh38'
    return row

def extract_rsid(s):
    import re
    match = re.findall(r"rs\d+", s)
    return match[-1] if match else None

import requests

def get_rsid_position(rsid, assembly="GRCh38"):
    url = f"https://rest.ensembl.org/variation/human/{rsid}"
    headers = {"Content-Type": "application/json"}
    response = requests.get(url, headers=headers, params={"genome": assembly})
    if response.status_code == 200:
        data = response.json()
        for mapping in data.get("mappings", []):
            if mapping["assembly_name"] == assembly:
                return {
                    "chromosome": mapping["seq_region_name"],
                    "start": mapping["start"],
                    "end": mapping["end"],
                }
    return None



def get_start_end_strand_control(row):
    """
    Special cases to set chr, start, end and strand for control sequences from their header (because not in region bed)

    Case: C_negative_neuron_MK: (all C_negative_neuron_MK sequences have "_chr" pattern)
            header: C_negative_neuron_MK:tile_14444_chr15_67066278_67066547_reference__1.1385203581298
                => chr: 15, start: 67066278, end: 67066547, strand: . (no information given)
            variant_header: >C_negative_neuron_MK:tile_36043_chr6_14500968_14501237_G_C_261__0.274168942409752

    Case: C_positive_neuron_MK: (all C_positive_neuron_MK sequences have "_chr" pattern")
            header: C_positive_neuron_MK:tile_35742_chr6_3247831_3248100_reference_0.892141141777512
                => chr: 6, start: 3247831, end: 3248100, strand: . (no information given)

    Case: C_negative_heart_MK: ( all C_negative_heart_MK sequences have "_chr" pattern)
            header: C_negative_heart_MK:tile_6903_chr11_9614045_9614314_reference__0.958461950470297
                => chr: 11, start: 9614045, end: 9614314, strand: . (no information given)

    Case: C_positive_heart_MK: (all C_positive_heart_MK sequences have "_chr" pattern)
            header: C_positive_heart_MK:tile_7939_chr11_65487592_65487861_reference_1.25449216981846
                => chr: 11, start: 65487592, end: 65487861, strand: . (no information given)

    Case: C_negative_neuron_NP: (all C_negative_neuron_NP sequences have "_chr" pattern)
            header: C_negative_neuron_NP:GW18_PFC_ABC_chr15_89400286_89400556_0.830617698776558
                => chr: 15, start: 89400286, end: 89400556, strand: . (no information given)
            additional condition: scrambled_control____2.28928058620308
                => category: scrambled

    Case: C_positive_neuron_NP: (all C_positive_neuron_NP sequences have "_chr" pattern)
            header: C_positive_neuron_NP:GW18_PFC_ABC_chr11_65487667_65487937_5.27702983385667
                => chr: 11, start: 65487667, end: 65487937, strand: . (no information given)

    Case: C_positive_neuron_CD: headers have "::chr" pattern and delimited by "-mean_ratio"
            header: C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A::chr4:155359187-155359457-mean_ratio2.42
                => chr: 4, start: 155359187, end: 155359457, strand: . (no information given)
            additional condition: C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_94-mean_ratio2.31

    """

    if row[col_category] == 'scrambled':
        row[col_class] = "element inactive control"
        return row
    name = row[col_name]

    if name.startswith('C_negative_neuron_MK') or name.startswith('C_positive_neuron_MK'):
        row[col_source] = 'Michael Kosicki'
        row[col_class] = 'element inactive control'
        if 'scramble' in name:
            row[col_category] = 'scrambled'
            return row

        if 'positive_neuron' in name:
            row[col_class] = 'element active control'
        region_info = '_'.join(name.split('_chr')[1].split('_')[:3]) # 11_9614045_9614314
        row[col_category] = 'element'
        row[col_chr] = f"chr{region_info.split('_')[0]}"
        row[col_start] = int(region_info.split('_')[1]) -1 # turn into 0-based
        row[col_end] = int(region_info.split('_')[2])
        row[col_strand] = '+'
        if len(name.split('_chr')[1].split("_")) == 7:
            row[col_variant_class] = 'SNV'
            row[col_category] = 'variant'
            row[col_class] = 'variant negative control'
            row[my_col_ref_base] = name.split('_chr')[1].split("_")[3]
            row[my_col_alt_base] = name.split('_chr')[1].split("_")[4]
            row[col_variant_pos] = int(name.split('_chr')[1].split("_")[5]) - 1
            row[col_allele] = 'alt'
            if 'positive_neuron' in name:
                row[col_class] = 'variant positive control'

    elif name.startswith('C_negative_neuron_NP') or name.startswith('C_positive_neuron_NP'):
        row[col_class] = 'element inactive control'
        row[col_source] = 'Nick Page'

        if 'scramble' in name:
            row[col_category] = 'scrambled'
            return row
        if 'positive_neuron' in name:
            row[col_class] = 'element active control'
        region_info = '_'.join(name.split('_chr')[1].split('_')[:3]) # 11_9614045_9614314
        # print(region_info)
        row[col_chr] = f"chr{region_info.split('_')[0]}"
        row[col_start] = int(region_info.split('_')[1])
        row[col_end] = int(region_info.split('_')[2])
        row[col_strand] = '+'

    elif name.startswith('C_positive_neuron_CD'):
        # note the given positions of the variants are given as 50% but sometimes it is n/2 and sometimes n/2+1 => rsid to position
        # used the explanation from Chengyu mail: 10.12.2024
        row[col_class] = 'element inactive control'
        row[col_source] = 'Chengyu Deng'

        if 'NA_NA_NA' in name:
            row[col_category] = 'scrambled'
            row[col_chr] = 'NA'
            row[col_start] = 'NA'
            row[col_end] = 'NA'
            row[col_strand] = 'NA'
            return row
        row[col_class] = 'element active control'
        region_info = name.split('::chr')[1].split('-mean_ratio')[0] # 4:155359187-155359457
        row[col_chr] = f"chr{region_info.split(':')[0]}"
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '+'

        variant_info = name.split('C_positive_neuron_CD:')[1].split('::chr')[0].split('_')
        rsid = extract_rsid(name) # e.g. p1_rs10061048_A_G_ref_50_A::chr5:1030722-1030992-mean_ratio2.09
        position = get_rsid_position(rsid)
        if position:
            row[col_variant_class] = 'SNV'
            row[col_category] = 'variant'
            row[col_class] = 'variant positive control'
            row[col_allele] = 'alt'
            row[my_col_ref_base] = variant_info[2]
            row[my_col_alt_base] = variant_info[3]
            row[col_variant_pos] = int(position['start']) - 1 - row[col_start] # 1-based start (=> 0-based location should be start - 1)
            if '_ref_' in name:
                row[col_allele] = 'ref'
                reference_info = 'possible reference but no alternative assigned'
                row[col_variant_pos] = int(position['start']) - 1 - row[col_start]
                row[my_col_alt_base] = 'NA'
                row[my_col_ref_base] = 'NA'

        else:
            print('Rsid not found: ', rsid)
    return row


# dict of chr number to refseq chromosome number
chrom_2_refseq = {
    "chr1": "NC_000001.11",
    "chr2": "NC_000002.12",
    "chr3": "NC_000003.12",
    "chr4": "NC_000004.12",
    "chr5": "NC_000005.10",
    "chr6": "NC_000006.12",
    "chr7": "NC_000007.14",
    "chr8": "NC_000008.11",
    "chr9": "NC_000009.12",
    "chr10": "NC_000010.11",
    "chr11": "NC_000011.10",
    "chr12": "NC_000012.12",
    "chr13": "NC_000013.11",
    "chr14": "NC_000014.9",
    "chr15": "NC_000015.10",
    "chr16": "NC_000016.10",
    "chr17": "NC_000017.11",
    "chr18": "NC_000018.10",
    "chr19": "NC_000019.10",
    "chr20": "NC_000020.11",
    "chr21": "NC_000021.9",
    "chr22": "NC_000022.11",
    "chrX": "NC_000023.11",
    "chrY": "NC_000024.10"}


def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) # I think everything is now 0-based
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'


def get_spdi_chengyu_header(row):
    """
    Returns the SPDI identifier for the given variant using start and variant_pos
    Assumption: allele need to be set beforehands
    """
    if not (row[my_col_ref_base] and row[my_col_alt_base]):
        row['SPDI'] = 'NA'
        return row
    # identify the variant chrom-pos-ref-alt pattern
    chrom_pos_ref_alt = f'{row[col_chr]}-{row[col_start]+row[col_variant_pos]}-{row[my_col_ref_base]}-{row[my_col_alt_base]}'
    if chrom_pos_ref_alt == "NA":
        raise ValueError('Variant pattern could not be found')
    # create the SPDI identifier
    row[col_SPDI] = create_speedy_chromosomes(chrom_pos_ref_alt, seperator='-', indices=[0,1,2,3])
    return row


In [36]:
input_fasta = config['design_file']
pre_metadata_df = hf.fasta_to_dataframe(input_fasta, columns=[col_name, col_sequence])
pre_metadata_df

# split the metadata file headers by '#'
# Apply the function to each row and concatenate the results
pre_metadata_df_split = pd.concat(pre_metadata_df.apply(lambda row: hf.split_ids(row, id_col=col_name, separator='#'), axis=1).values)

# Reset the index
pre_metadata_df_split.reset_index(drop=True, inplace=True)

pre_metadata_df = pre_metadata_df_split.copy()
pre_metadata_df['tmp_label'] = pre_metadata_df[col_name].apply(lambda x: hf.get_label(x))
print(pre_metadata_df.shape[0]) # 80803

80803


In [37]:
underscore_parsable_headers = ['C_negative_neuron_NP', 'C_positive_neuron_NP', 'C_positive_neuron_MK',  'C_negative_neuron_MK', 'C_positive_neuron_CD']

# # filter for underscore parsable headers
pre_metadata_df_filtered = pre_metadata_df.loc[pre_metadata_df['tmp_label'].isin(underscore_parsable_headers)].copy()
print(pre_metadata_df_filtered.shape[0]) # 746

# add the columns of the metadata file
pre_metadata_df_filtered[col_category] = 'NA'
pre_metadata_df_filtered[col_class] = 'NA'
pre_metadata_df_filtered[col_source] = 'NA'
pre_metadata_df_filtered[col_ref] = 'NA'
pre_metadata_df_filtered[col_chr] = 'NA'
pre_metadata_df_filtered[col_start] = 'NA'
pre_metadata_df_filtered[col_end] = 'NA'
pre_metadata_df_filtered[col_strand] = 'NA'
pre_metadata_df_filtered[col_variant_class] = 'NA'
pre_metadata_df_filtered[col_variant_pos] = 'NA'
pre_metadata_df_filtered[col_SPDI] = 'NA'
pre_metadata_df_filtered[col_allele] = 'NA'
pre_metadata_df_filtered[col_info] = ''


pre_metadata_df_filtered[col_category] = pre_metadata_df_filtered[col_name].apply(get_category)
pre_metadata_df_filtered = pre_metadata_df_filtered.apply(get_reference_genome, axis=1)

# parse regions from header
pre_metadata_df_filtered = pre_metadata_df_filtered.apply(get_start_end_strand_control, axis=1)
pre_metadata_df_filtered

# add SPDI
pre_metadata_df_filtered = pre_metadata_df_filtered.apply(lambda row: get_spdi(row), axis=1)

# remove adapter from sequence (15bp of start and end):
pre_metadata_df_filtered[col_sequence] = pre_metadata_df_filtered[col_sequence].apply(lambda x: x[15:-15])


746


In [38]:
# function to generate arrays out off the columns
def make_column_arrays(row):
    """
    create arrays for the required columns
    """
    allele = row[col_allele]
    SPDI = row[col_SPDI]
    variant_pos = row[col_variant_pos]
    variant_class = row[col_variant_class]

    if allele == 'ref' or allele == 'alt': # only "alt" is string
        row[col_allele] = [allele]
    if isinstance(SPDI, str): # only for alt sequences this is true
        if SPDI != "NA":
            row[col_SPDI] = [SPDI]
    if isinstance(variant_pos, float):
        row[col_variant_pos] = [int(variant_pos)]
    elif isinstance(variant_pos, int):
        row[col_variant_pos] = [int(variant_pos)]
    if isinstance(variant_class, str):
        if row[col_variant_class] in ['SNV', 'indel']:
                row[col_variant_class] = [variant_class]
    if not isinstance(row[col_class], str):
        print(row[col_name])
    return row

In [39]:
pre_metadata_df_filtered = pre_metadata_df_filtered.apply(make_column_arrays, axis=1)

In [40]:
pre_metadata_df_filtered.loc[pre_metadata_df_filtered[col_class] == "NA"]

,SPDI,allele,category,chr,class,end,info,name,ref,sequence,source,start,strand,tmp_alt_base,tmp_label,tmp_ref_base,variant_class,variant_pos


In [41]:
interesting_columns = [col_name, col_sequence, col_category, col_class, col_source, col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]
group_name = 'neuro_controls'
output_dir = config['final_output_dir']
output_path = os.path.join(output_dir, group_name)
for group_name in underscore_parsable_headers:
    pre_metadata_df_filtered_group = pre_metadata_df_filtered.loc[pre_metadata_df_filtered['tmp_label'] == group_name].copy()
    pre_metadata_df_group = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == group_name].copy()
    print(group_name)
    print(f'Expected: {pre_metadata_df_group.shape[0]}', )
    print(f'Actuall number: {pre_metadata_df_filtered_group.shape[0]}')
    # show the duplicated sequences
    print(f'Unique header number: {pre_metadata_df_filtered_group[col_sequence].nunique()}')
    # Write DataFrame to TSV file
    pre_metadata_df_filtered_group[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
    os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')


C_negative_neuron_NP
Expected: 217
Actuall number: 217
Unique header number: 217
C_positive_neuron_NP
Expected: 99
Actuall number: 99
Unique header number: 99
C_positive_neuron_MK
Expected: 100
Actuall number: 100
Unique header number: 100
C_negative_neuron_MK
Expected: 234
Actuall number: 234
Unique header number: 234
C_positive_neuron_CD
Expected: 96
Actuall number: 96
Unique header number: 96


In [42]:
# not useful
# group_name = 'neuro_controls'
# output_dir = config['final_output_dir']
# output_path = os.path.join(output_dir, group_name)
# # Write DataFrame to TSV file
# pre_metadata_df_filtered[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
# os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')
# # Write DataFrame to TSV file
# pre_metadata_df_filtered[interesting_columns].to_csv('/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/neuro_controls/neuro_controls.metadata.tmp.tsv.gz', sep='\t', index=False, na_rep='NA', compression='gzip')
# import os
# os.system('zcat /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/neuro_controls/neuro_controls.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/neuro_controls/neuro_controls.metadata.tsv.gz')

In [43]:
breaking spot

SyntaxError: invalid syntax (2270176476.py, line 1)

### Working on C_positive_neuron_CD (errors in the header derived SPDIs)

In [ ]:
chengyu_names_df = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == 'C_positive_neuron_CD'].copy()
chengyu_names_df['no_label'] = chengyu_names_df[col_name].apply(lambda name: name.replace('C_positive_neuron_CD:', ''))
# chengyu_names_df[col_sequence] = chengyu_names_df[col_sequence].apply(lambda x: x[15:-15])
# chengyu_names_df[col_sequence].to_list()
chengyu_names_df

chengyu_names_df['no_label'].to_list()
# hf.write_fasta(chengyu_names_df, '80K_MPRA_group_C_positive_neuron_CD.fa', header=['no_label', 'sequence'])
# rs55985730: from dbsnp: NC_000007.14:g.128776990T>G 128776855 + 135 = 128776990
# rs9975055: from dbsnp: NC_000021.9:44930103:T:G 44929969 + 135 = 44930104

['c1_NA_NA_NA_NA::72hr_top_99-mean_ratio1.92',
 'c1_NA_NA_NA_NA::72hr_top_4-mean_ratio3.08',
 'p1_rs6813360_A_C_ref_50_A::chr4:155359187-155359457-mean_ratio2.42',
 'p1_rs55985730_T_G_alt_50_T::chr7:128776855-128777125-mean_ratio2.34',
 'c1_NA_NA_NA_NA::72hr_top_94-mean_ratio2.31',
 'p1_rs7115714_G_A_ref_50_A::chr11:120424017-120424287-mean_ratio2.22',
 'p1_rs9975055_T_G_alt_50_G::chr21:44929969-44930239-mean_ratio2.18',
 'n1_rs2279982_G_A_alt_50::chr2:164841902-164842172-mean_ratio2.16',
 'p1_rs34761481_G_A_alt_50_G::chr7:129161739-129162009-mean_ratio2.14',
 'p1_rs7115714_G_A_alt_50_A::chr11:120424017-120424287-mean_ratio2.13',
 'n1_rs275835_G_A_alt_50::chr7:132509137-132509407-mean_ratio2.11',
 'p1_rs10061048_A_G_ref_50_A::chr5:1030722-1030992-mean_ratio2.09',
 'p1_rs114772924_G_A_ref_50_A::chr1:32253766-32254036-mean_ratio2.08',
 'p1_rs11757302_C_T_ref_50_C::chr6:905837-906107-mean_ratio2.06',
 'p1_rs115202710_C_T_alt_50_C::chr6:24704205-24704475-mean_ratio2.05',
 'p1_rs62086577_G_

In [8]:
metadata_df_path = "/data/cephfs-2/unmirrored/groups/kircher/IGVF_data_submssion_TM/80K/final_design/MPRA_250326.metadata.tsv.gz"
# multiname:
# GC_GABA_Chengyu:GABA|chr13:86882257-86882526|+|1.63;C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_99-mean_ratio1.92;MK:tile_11028|chr13-86882257+86882526|reference
# C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_94-mean_ratio2.31;MK:tile_32319|chr4-165327322+165327591|reference
# C_positive_heart_MK:tile_7939_chr11_65487592_65487861_reference_1.25449216981846;C_positive_neuron_MK:tile_7939_chr11_65487592_65487861_reference_1.25449216981846;C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_4-mean_ratio3.08;MK:tile_7939|chr11-65487592+65487861|reference

In [30]:
import ast
# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'


list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]

# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

def extract_rsid(s):
    import re
    match = re.findall(r"rs\d+", s)
    return match[-1] if match else None


metadata_df = pd.read_csv(metadata_df_path, sep="\t", low_memory=False)

# Apply the safe_eval function to the specified columns
for col in list_columns:
    metadata_df[col] = metadata_df[col].apply(safe_eval)


In [ ]:
chengyu_names_df = metadata_df.loc[metadata_df[col_name].str.contains("C_positive_neuron_CD")].copy()
chengyu_names_df['RSIDs'] = chengyu_names_df[col_name].apply(extract_rsid)
chengyu_names_df

In [32]:
chengyu_names_df

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info
77626,C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A...,GATTGTATAAAGAAAATGTGTATACACACACACACACACACACACA...,variant,variant positive control,Chengyu Deng,GRCh38,chr4,155359187.0,155359457.0,+,[SNV],[134],None,[ref],NaN
77627,C_positive_neuron_CD:p1_rs55985730_T_G_alt_50_...,GGCGCCTGTAGTCCCAGCTACTTGGGAGGCTGAGGCAGGAGAATGG...,variant,variant positive control,Chengyu Deng,GRCh38,chr7,128776855.0,128777125.0,+,[SNV],[134],[NC_000007.14:128776989:T:G],[alt],NaN
77628,C_positive_neuron_CD:p1_rs7115714_G_A_ref_50_A...,TGGTAATTAAAAGCAAAGAGATCTTTTCTATTTGTATGAGCCCTTC...,variant,variant positive control,Chengyu Deng,GRCh38,chr11,120424017.0,120424287.0,+,[SNV],[134],None,[ref],NaN
77629,C_positive_neuron_CD:p1_rs9975055_T_G_alt_50_G...,CCCTGCTCCCCAGTTCCCACCAGAAACCCCAAGTGGGTGTTCCAGC...,variant,variant positive control,Chengyu Deng,GRCh38,chr21,44929969.0,44930239.0,+,[SNV],[134],[NC_000021.9:44930103:T:G],[alt],NaN
77630,C_positive_neuron_CD:n1_rs2279982_G_A_alt_50::...,TGCGGGCGCTGGCTGGGCGCTGGGGGCCTCGCTGGAGCCCGCTCTC...,variant,variant positive control,Chengyu Deng,GRCh38,chr2,164841902.0,164842172.0,+,[SNV],[134],[NC_000002.12:164842036:G:A],[alt],NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77717,C_positive_neuron_CD:p2_rs9926320_G_A_ref_25_A...,GTTTGCAGAGTGGGTGTGGTGCCGGAGCACAGGAGCACCTCTTCTG...,variant,variant positive control,Chengyu Deng,GRCh38,chr16,69094082.0,69094352.0,+,[SNV],[67],None,[ref],NaN
77718,C_positive_neuron_CD:n1_rs7214382_G_C_alt_50::...,GGACATCCCCAGGGACCCCACCAGCCCGGCCCGTAGCCCAGCGGTG...,variant,variant positive control,Chengyu Deng,GRCh38,chr17,33292592.0,33292862.0,+,[SNV],[134],[NC_000017.11:33292726:G:C],[alt],NaN
80154,GC_GABA_Chengyu:GABA|chr13:86882257-86882526|+...,CCTTTCCTTCTTACTAGGTGTACCAGGTGTCATGGCCACCTCCAGA...,element,element inactive control,NaN,GRCh38,chr13,86882256.0,86882526.0,NaN,None,None,None,None,NaN
80158,C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_...,CTCCCCCGCACAGCGCAGGCTCTCACTGGGAATCTGCCGGGACCGC...,scrambled,element inactive control,Chengyu Deng,GRCh38,NaN,NaN,NaN,NaN,None,None,None,None,NaN


In [33]:
chengyu_names_df['RSIDs'] = chengyu_names_df[col_name].apply(extract_rsid)
chengyu_names_df

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,RSIDs
77626,C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A...,GATTGTATAAAGAAAATGTGTATACACACACACACACACACACACA...,variant,variant positive control,Chengyu Deng,GRCh38,chr4,155359187.0,155359457.0,+,[SNV],[134],None,[ref],NaN,rs6813360
77627,C_positive_neuron_CD:p1_rs55985730_T_G_alt_50_...,GGCGCCTGTAGTCCCAGCTACTTGGGAGGCTGAGGCAGGAGAATGG...,variant,variant positive control,Chengyu Deng,GRCh38,chr7,128776855.0,128777125.0,+,[SNV],[134],[NC_000007.14:128776989:T:G],[alt],NaN,rs55985730
77628,C_positive_neuron_CD:p1_rs7115714_G_A_ref_50_A...,TGGTAATTAAAAGCAAAGAGATCTTTTCTATTTGTATGAGCCCTTC...,variant,variant positive control,Chengyu Deng,GRCh38,chr11,120424017.0,120424287.0,+,[SNV],[134],None,[ref],NaN,rs7115714
77629,C_positive_neuron_CD:p1_rs9975055_T_G_alt_50_G...,CCCTGCTCCCCAGTTCCCACCAGAAACCCCAAGTGGGTGTTCCAGC...,variant,variant positive control,Chengyu Deng,GRCh38,chr21,44929969.0,44930239.0,+,[SNV],[134],[NC_000021.9:44930103:T:G],[alt],NaN,rs9975055
77630,C_positive_neuron_CD:n1_rs2279982_G_A_alt_50::...,TGCGGGCGCTGGCTGGGCGCTGGGGGCCTCGCTGGAGCCCGCTCTC...,variant,variant positive control,Chengyu Deng,GRCh38,chr2,164841902.0,164842172.0,+,[SNV],[134],[NC_000002.12:164842036:G:A],[alt],NaN,rs2279982
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77717,C_positive_neuron_CD:p2_rs9926320_G_A_ref_25_A...,GTTTGCAGAGTGGGTGTGGTGCCGGAGCACAGGAGCACCTCTTCTG...,variant,variant positive control,Chengyu Deng,GRCh38,chr16,69094082.0,69094352.0,+,[SNV],[67],None,[ref],NaN,rs9926320
77718,C_positive_neuron_CD:n1_rs7214382_G_C_alt_50::...,GGACATCCCCAGGGACCCCACCAGCCCGGCCCGTAGCCCAGCGGTG...,variant,variant positive control,Chengyu Deng,GRCh38,chr17,33292592.0,33292862.0,+,[SNV],[134],[NC_000017.11:33292726:G:C],[alt],NaN,rs7214382
80154,GC_GABA_Chengyu:GABA|chr13:86882257-86882526|+...,CCTTTCCTTCTTACTAGGTGTACCAGGTGTCATGGCCACCTCCAGA...,element,element inactive control,NaN,GRCh38,chr13,86882256.0,86882526.0,NaN,None,None,None,None,NaN,None
80158,C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_...,CTCCCCCGCACAGCGCAGGCTCTCACTGGGAATCTGCCGGGACCGC...,scrambled,element inactive control,Chengyu Deng,GRCh38,NaN,NaN,NaN,NaN,None,None,None,None,NaN,None


In [58]:
chengyu_names_df['sequence'].nunique()

96

In [ ]:
GATTGTATAAAGAAAATGTGTATACACACACACACACACACACACACACACACACACACAGGAATACTATTCAGCCACACAGAAAAGAAATAATATCTTTTACAACATGGATGGAACTGGAGGCTATTACCTTAAGTGAAATAAATCAGAAACAGAAAGTCAAATGGTACATGTTCTCACTTACAAGTCAGAGCTAAATAATGTGTACACATGTATATAGGGTATGGAATAATAGATAATGGAGACTAGGAAAGGTGAGAGGGAGGTGCG

NameError: name 'CATTAAAAAGCAATCACTTATCAAAACGTCCGCGGGGGCGGGGGTAGAATGTGGGAGGTGCGGGAGCACCCCTTCCCCATGACAGCCCGGTCATCGAACTTCAGCCCCCACCCAGGACCATCAAGACTGCGCCTCCCCTCCCCACATCCTACCAAGGTTAGGAAGATAGGAGCACCCACAGTTCTGAGGGATAAAAATGA' is not defined

In [ ]:
# Get their genomic coordintates from hg38 by mapping to the reference genome
# hg38 on the cubi cluster: /data/cephfs-1/work/projects/cubit/current/static_data/reference/hg38/ucsc/hg38.fa

# Do I have to every alt a ref?
# C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A::chr4:155359187-155359457-mean_ratio2.42
chengyu_names_df_no_alt = chengyu_names_df.loc[~chengyu_names_df[col_name].str.contains("_alt_")].copy()
chengyu_names_df_no_alt

# get their spdis from the rsids (I have a function for this already)
chengyu_names_df_ref = chengyu_names_df_no_alt.loc[~chengyu_names_df_no_alt[col_name].str.contains("_NA_")].copy()
chengyu_names_df_ref

# extract the rsid

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,RSIDs
77626,C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A...,GATTGTATAAAGAAAATGTGTATACACACACACACACACACACACA...,variant,variant positive control,Chengyu Deng,GRCh38,chr4,155359187.0,155359457.0,+,[SNV],[134],None,[ref],NaN,rs6813360
77628,C_positive_neuron_CD:p1_rs7115714_G_A_ref_50_A...,TGGTAATTAAAAGCAAAGAGATCTTTTCTATTTGTATGAGCCCTTC...,variant,variant positive control,Chengyu Deng,GRCh38,chr11,120424017.0,120424287.0,+,[SNV],[134],None,[ref],NaN,rs7115714
77634,C_positive_neuron_CD:p1_rs10061048_A_G_ref_50_...,TCGGGACTGCAGCCCCCAGGAGGGAGGCCTACTTTTTAATAATCAG...,variant,variant positive control,Chengyu Deng,GRCh38,chr5,1030722.0,1030992.0,+,[SNV],[134],None,[ref],NaN,rs10061048
77635,C_positive_neuron_CD:p1_rs114772924_G_A_ref_50...,AGTGTGAGAACTCAGGCAGGCTGGACTCTGAGCGGGGCCATGGGAT...,variant,variant positive control,Chengyu Deng,GRCh38,chr1,32253766.0,32254036.0,+,[SNV],[134],None,[ref],NaN,rs114772924
77636,C_positive_neuron_CD:p1_rs11757302_C_T_ref_50_...,CACTGTGAACTGCAAGAGAGGCAGGAAATGTAGCTGAGCACACAGT...,variant,variant positive control,Chengyu Deng,GRCh38,chr6,905837.0,906107.0,+,[SNV],[134],None,[ref],NaN,rs11757302
77640,C_positive_neuron_CD:p1_rs66500423_T_C_ref_50_...,CCACTCACACACAGTCACAACTCCACCCACGTCCTGTCAGTCACAC...,variant,variant positive control,Chengyu Deng,GRCh38,chr19,40689130.0,40689400.0,+,[SNV],[134],None,[ref],NaN,rs66500423
77642,C_positive_neuron_CD:n1_rs12773142_C_G_ref_50:...,CGGCCCGCCCCTCGGCCCGCGCGGCCATTGTCTGCGCCAGGCGCCG...,variant,variant positive control,Chengyu Deng,GRCh38,chr10,78303207.0,78303477.0,+,[SNV],[134],None,[ref],NaN,rs12773142
77643,C_positive_neuron_CD:p1_rs6916842_G_T_ref_50_T...,AGTCACTGCGCCTGGCCCAAAAAATGTGTGTGTGTTTTTTCCTTTT...,variant,variant positive control,Chengyu Deng,GRCh38,chr6,116747354.0,116747624.0,+,[SNV],[134],None,[ref],NaN,rs6916842
77645,C_positive_neuron_CD:n1_rs275835_G_A_ref_50::c...,CGCAAAGACTGGCTGGAGGAGAAGAGGGAGGGAGGGAGGGAGGGAG...,variant,variant positive control,Chengyu Deng,GRCh38,chr7,132509137.0,132509407.0,+,[SNV],[134],None,[ref],NaN,rs275835
77646,C_positive_neuron_CD:p1_rs2293578_C_T_ref_50_T...,AGTGAGAGCAGCCCAAGGATCCAGGGTGCAGGGAACTCCAGAGCTG...,variant,variant positive control,Chengyu Deng,GRCh38,chr11,47415717.0,47415987.0,+,[SNV],[134],None,[ref],NaN,rs2293578


In [ ]:
155359188+270

155359458

In [ ]:
# C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A::chr4:155359187-155359457-mean_ratio2.42

In [20]:
chengyu_names_df_alt.columns

NameError: name 'chengyu_names_df_alt' is not defined

In [ ]:
# chengyu alt:
chengyu_names_df_alt = chengyu_names_df.loc[chengyu_names_df[col_name].str.contains("_alt_")].copy()
chengyu_names_df_alt

# extract the rsids
rsid_set_alt = set(chengyu_names_df_alt['RSIDs'].dropna().unique())
rsid_set_ref = set(chengyu_names_df_ref['RSIDs'].dropna().unique())

rsid_set_alt.intersection(rsid_set_ref)

# {'rs275835',
#  'rs34241773',
#  'rs606742',
#  'rs62086577',
#  'rs7115714',
#  'rs76990668',
#  'rs9931091'}
# rsid_set_alt - rsid_set_ref

{'rs275835',
 'rs34241773',
 'rs606742',
 'rs62086577',
 'rs7115714',
 'rs76990668',
 'rs9931091'}

In [ ]:
# align them to the reference genome:
import subprocess
import pysam
import tempfile


In [ ]:
# bwa index -p /data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/hg38/hg38 /data/cephfs-1/work/projects/cubit/current/static_data/reference/hg38/ucsc/hg38.fa

In [ ]:
import pandas as pd
import subprocess
import os
import tempfile
import pysam
import ast


# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'


list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]

# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

def extract_rsid(s):
    import re
    match = re.findall(r"rs\d+", s)
    return match[-1] if match else None


import re # Import regex module

def extract_variant_info(read: pysam.AlignedSegment):
    """
    Extracts single nucleotide variant (SNV) information from a pysam AlignedSegment.
    Assumes a single mismatch based on NM:i:1 and a simple MD:Z tag format.

    Args:
        read (pysam.AlignedSegment): A pysam AlignedSegment object representing
                                     a mapped read.

    Returns:
        tuple: (variant_pos_in_seq, ref_allele, alt_allele)
               Returns (None, None, None) if no single SNV is found or
               if the MD tag format is not as expected for a simple SNV.
    """
    variant_pos_in_seq = None
    ref_allele = None
    alt_allele = None

    # Check if the read has exactly one mismatch (NM tag)
    if read.has_tag('NM') and read.get_tag('NM') == 1:
        # Check if the read has the MD tag
        if read.has_tag('MD'):
            md_tag = read.get_tag('MD')

            # Regex to parse MD:Z: tag for a single SNV
            # Example: MD:Z:202A67 -> (202, A, 67)
            # This regex captures:
            # Group 1: number of matches before mismatch (e.g., '202')
            # Group 2: reference base at mismatch (e.g., 'A')
            # Group 3: number of matches after mismatch (e.g., '67')
            match = re.match(r'^(\d+)([ACGTN])(\d*)$', md_tag)

            if match:
                # Extract components
                matches_before = int(match.group(1))
                ref_base = match.group(2)

                # The variant position in the query sequence is 0-based,
                # which is simply the number of matches before the mismatch.
                variant_pos_in_seq = matches_before
                ref_allele = ref_base

                # Get the alternative allele from the query sequence at that position
                if read.query_sequence and variant_pos_in_seq < len(read.query_sequence):
                    alt_allele = read.query_sequence[variant_pos_in_seq]
                else:
                    print(f"Warning: Could not get alt allele for read {read.query_name} at pos {variant_pos_in_seq}. Query sequence might be missing or position out of bounds.")
                    alt_allele = "N" # Fallback
            else:
                # MD tag format is not a simple single SNV (e.g., contains indels, multiple mismatches)
                # print(f"Debug: MD tag '{md_tag}' for read {read.query_name} does not match simple SNV pattern.")
                # raise error:
                raise ValueError(f"MD tag '{md_tag}' for read {read.query_name} does not match simple SNV pattern.")
                pass
    # else:
        # print(f"Debug: Read {read.query_name} has NM tag {read.get_tag('NM')} != 1 or no NM tag.")

    return variant_pos_in_seq, ref_allele, alt_allele



def get_genomic_coordinates_bed(dataframe: pd.DataFrame,
                                 sequence_column_name: str,
                                 hg38_fa_path: str,
                                 output_bed_path: str,
                                 id_column_name: str = None,
                                 output_sam_path: str = None,
                                 output_id_map_path: str = None):
    """
    Finds genomic coordinates for sequences in a DataFrame using BWA,
    writes them to a BED file, and optionally saves the SAM alignment file.

    Args:
        dataframe (pd.DataFrame): The input DataFrame containing sequences.
        sequence_column_name (str): The name of the column in the DataFrame
                                    that contains the sequences.
        hg38_fa_path (str): Path to the hg38 reference FASTA file (e.g., 'hg38.fa').
        output_bed_path (str): Path where the output BED file will be written.
        id_column_name (str, optional): The name of the column to use as the
                                        name field in the BED file. If None,
                                        DataFrame index will be used.
        output_sam_path (str, optional): Path where the SAM alignment file
                                         will be saved. If None, a temporary
                                         SAM file is used and cleaned up.
    """

    temp_fasta_path = None
    temp_sam_path_internal = None # Internal variable for temporary SAM file path
    sam_file_for_pysam = None     # Path that pysam will actually read from
    id_mapping = {}               # Dictionary to store short_id -> original_id mapping
    # 1. Create a temporary FASTA file from DataFrame sequences
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.fa') as temp_fasta:
        temp_fasta_path = temp_fasta.name
        for idx, row in dataframe.iterrows():
            # Get the original sequence ID
            original_seq_id = str(row[id_column_name]) if id_column_name else f"df_idx_{idx}"

            # Generate a short, unique ID for the FASTA/SAM/BED files
            # This ensures compatibility with SAM/BAM query name length limits
            short_seq_id = f"seq_{idx}"

            # Store the mapping from short ID to original ID
            id_mapping[short_seq_id] = original_seq_id

            sequence = row[sequence_column_name]
            if not isinstance(sequence, str):
                print(f"Warning: Skipping non-string sequence at index {idx}: {sequence}")
                continue
            temp_fasta.write(f">{short_seq_id}\n{sequence}\n")
    print(f"Temporary FASTA created at: {temp_fasta_path}")
    print(f"Generated {len(id_mapping)} short IDs and stored mapping.")

    # Determine BWA index prefix based on bwa_index_dir
    bwa_ref_prefix = hg38_fa_path # Default to FASTA path

    # 2. Check for BWA index and create if not present
    # BWA index files: .amb, .ann, .bwt, .pac, .sa
    bwa_index_suffixes = [".amb", ".ann", ".bwt", ".pac", ".sa"]
    bwa_index_files = [f"{bwa_ref_prefix}{suffix}" for suffix in bwa_index_suffixes]

    index_exists = all(os.path.exists(f) for f in bwa_index_files)

    if not index_exists:
        print(f"BWA index for {hg38_fa_path} not found at {bwa_ref_prefix}. Creating index...")
        try:
            # Use subprocess to run bwa index command with -p for prefix
            subprocess.run(['bwa', 'index', '-p', bwa_ref_prefix, hg38_fa_path],
                            check=True,
                            capture_output=True,
                            text=True)
            print("BWA index created successfully.")
        except subprocess.CalledProcessError as e:
            print(f"Error creating BWA index: {e.stderr}")
            raise
    else:
        print(f"BWA index for {hg38_fa_path} found at {bwa_ref_prefix}. Skipping indexing.")

    # 3. Run BWA alignment and capture SAM output
    print(f"Running BWA mem on sequences from {temp_fasta_path}...")
    bwa_command = ['bwa', 'mem', bwa_ref_prefix, temp_fasta_path] # Use the prefix for alignment
    try:
        bwa_process = subprocess.run(bwa_command,
                                        capture_output=True,
                                        text=True,
                                        check=True)
        sam_output = bwa_process.stdout
        print("BWA alignment completed.")
    except subprocess.CalledProcessError as e:
        print(f"Error running BWA mem. Stderr: {e.stderr}")
        raise

    # Determine where to write/read the SAM file
    if output_sam_path:
        # User wants to save the SAM file explicitly
        sam_file_for_pysam = output_sam_path
        with open(output_sam_path, 'w') as sam_file:
            sam_file.write(sam_output)
        print(f"SAM alignment saved to: {output_sam_path}")
    else:
        # Use a temporary SAM file
        with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.sam') as temp_sam_file:
            temp_sam_path_internal = temp_sam_file.name
            temp_sam_file.write(sam_output)
        sam_file_for_pysam = temp_sam_path_internal
        print(f"Temporary SAM created at: {temp_sam_path_internal}")

    # 4. Parse SAM output using pysam and prepare BED data
    bed_records = []
    # pysam.AlignmentFile can directly read from the SAM file (either temp or user-specified)
    with pysam.AlignmentFile(sam_file_for_pysam, "r") as samfile:
        for read in samfile.fetch(until_eof=True):
            # Skip unmapped reads and secondary/supplementary alignments
            if read.is_unmapped or read.is_secondary or read.is_supplementary:
                continue

            chrom = samfile.get_reference_name(read.reference_id)
            start = read.reference_start  # 0-based start
            end = read.reference_end      # 0-based exclusive end (matches BED end)

            # Get the short ID from the SAM file (read.query_name)
            short_name_from_sam = read.query_name
            # Look up the original ID using the mapping table
            # Use .get() with a fallback in case a short_name isn't found in mapping (shouldn't happen if logic is correct)
            original_name_for_bed = id_mapping.get(short_name_from_sam, short_name_from_sam)

            score = read.mapping_quality  # Mapping quality as score
            if read.is_reverse:
                raise ValueError("Reverse reads are not supported in this script. Please ensure all reads are forward-stranded.")
            strand = '+' if not read.is_reverse else '-'
            variant_pos, ref_allele, alt_allele = extract_variant_info(read)

            # Append all information to bed_records
            bed_records.append([
                chrom, start, end, original_name_for_bed, score, strand,
                variant_pos if variant_pos is not None else 'NA', # Use '.' for missing info
                ref_allele if ref_allele is not None else 'NA',
                alt_allele if alt_allele is not None else 'NA'
            ])

    # 5. Write BED file
    print(f"Writing BED file to {output_bed_path}...")
    with open(output_bed_path, 'w') as bed_file:
        for record in bed_records:
            bed_file.write('\t'.join(map(str, record)) + '\n')
    print(f"BED file successfully written to {output_bed_path}")

    # 6. Write ID mapping file if requested
    if output_id_map_path:
        id_map_df = pd.DataFrame(list(id_mapping.items()), columns=['short_id', 'original_id'])
        id_map_df.to_csv(output_id_map_path, index=False)
        print(f"ID mapping file saved to: {output_id_map_path}")

# Example Usage:
if __name__ == "__main__":
    metadata_df_path = "/data/cephfs-2/unmirrored/groups/kircher/IGVF_data_submssion_TM/80K/final_design/MPRA_250326.metadata.tsv.gz"

    metadata_df = pd.read_csv(metadata_df_path, sep="\t", low_memory=False)

    # Apply the safe_eval function to the specified columns
    for col in list_columns:
        metadata_df[col] = metadata_df[col].apply(safe_eval)

    chengyu_names_df = metadata_df.loc[metadata_df[col_name].str.contains("C_positive_neuron_CD")].copy()


    hg38_fa_path="/path/to/hg38.fa"  # Replace with your hg38 FASTA file path
    output_bed_file = "/path/to/output.bed"  # Replace with your desired
    hg38_fa_path = "/data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/hg38/hg38"
    output_bed_file = "/data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/C_positive_neuron_CD_hg38_coordinates.bed"
    output_sam_path = "/data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/C_positive_neuron_CD_hg38_alignment.sam"
    # --- Run the function ---
    try:
        get_genomic_coordinates_bed(
            dataframe=chengyu_names_df,
            sequence_column_name=col_sequence,
            hg38_fa_path=hg38_fa_path,
            output_bed_path=output_bed_file,
            id_column_name=col_name,
            output_sam_path=output_sam_path,
            output_id_map_path="/data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/name_mapping_table.tsv"
        )

    # ERROR: truncated file

    except Exception as e:
        print(f"\nAn error occurred during execution: {e}")

In [ ]:
# sam_path = "/data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/C_positive_neuron_CD_hg38_alignment.sam"
# import pysam
# bed_records = []
# # pysam.AlignmentFile can directly read from the SAM file (either temp or user-specified)
# with pysam.AlignmentFile(sam_path, "r") as samfile:
#     for read in samfile.fetch(until_eof=True):
#         # Skip unmapped reads and secondary/supplementary alignments
#         if read.is_unmapped or read.is_secondary or read.is_supplementary:
#             continue
#         # check if alignment 270M:
#         # print(read.query_name, read.cigarstring)
#         if read.cigarstring == '270M':
#             chrom = samfile.get_reference_name(read.reference_id)
#             start = read.reference_start  # 0-based start
#             end = read.reference_end      #  BED-like end
#             name = read.query_name        # Sequence ID from FASTA
#             score = read.mapping_quality  # Mapping quality as score
#             strand = '-' if read.is_reverse else '+'
#         else:
#             print(read.query_name, read.cigarstring)
#             break
#         bed_records.append([chrom, start, end, name, score, strand])

Notes: 
- 38 variants: (not perfect match)
- 1 variant: 
seq_77696       0       chr2    68162902        60      270M    *       0       0       ATTAGCCTGCAGTTGGGCAAAGTCATCCGACACAAATTCTACTTTATGACAGAGTGTTGAATATTGAATACTGTACTAAAAATGAAAAACAGAATGGTTGTATGTGTATTTGAAGTAGAGCATTCTGAATGTGTATCACCTTTGCACCCTTGTAAAGTTGAAAAATCCCAGTTCGAACCGTGTTAAGTTGGGAATCATCTGTGTACTTTTGGGTGAATAGATGGTCACCAACTTTGAAAGAGGCTTAAGTTGAGAAAATTTAAGAAATTA    *       NM:i:1  MD:Z:202A67     AS:i:265     XS:i:43

ATTAGCCTGCAGTTGGGCAAAGTCATCCGACACAAATTCTACTTTATGACAGAGTGTTGAATATTGAATACTGTACTAAAAATGAAAAACAGAATGGTTGTATGTGTATTTGAAGTAGAGCATTCTGAATGTGTATCACCTTTGCACCCTTGTAAAGTTGAAAAATCCCAGTTCGAACCGTGTTAAGTTGGGAATCATCTGTGTACTTTTGGGTGAATAGATGGTCACCAACTTTGAAAGAGGCTTAAGTTGAGAAAATTTAAGAAATTA

seq_77696,C_positive_neuron_CD:p2_rs7582466_A_G_alt_75_A::chr2:68162901-68163171-mean_ratio1.85
seq_77639       0       chr5    178613193       60      270M    *       0       0       GGGGCAATCACAAGGTCAGGAGATCGAGACCATCCTGGCTAACACTGTGAAATCCCATCTCTACTAAAAATACAAAAAATTAGCTGGGCATGGTGGTGGGCGCCTGTAGTCCCAGCTACTCGGGAGGCTGAGGCGGGAAAATGGCGTGAACCCGGGAGGCTGAGCTTGCTGGGACCCGAGAGCGCCACTGCACTCCAGCCTGGGTGACAGAGCGAGACTCCGTCTCAAAATAAATAAATAAATAAATAAATAAAAATAAAGATTTATCAA    *       NM:i:1  MD:Z:134A135    AS:i:265     XS:i:136


In [ ]:
# read the resulting bed:
genomic_region_bed_path = "/data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/C_positive_neuron_CD_hg38_coordinates.bed"
genomic_region_bed = pd.read_csv(genomic_region_bed_path, sep="\t", header=None)
genomic_region_bed.columns = [col_chr, col_start, col_end, col_name, 'score', col_strand, col_variant_pos, my_col_ref_base, my_col_alt_base]
genomic_region_bed

In [126]:
import pandas as pd
import subprocess
import os
import tempfile
import pysam
import ast
import re # Import regex module
import numpy as np


# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'


list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]

# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

def extract_rsid(s):
    import re
    match = re.findall(r"rs\d+", s)
    return match[-1] if match else None

def isNaN(num):
    return num != num


def extract_variant_info(read: pysam.AlignedSegment):
    """
    Extracts single nucleotide variant (SNV) information from a pysam AlignedSegment.
    Assumes a single mismatch based on NM:i:1 and a simple MD:Z tag format.

    Args:
        read (pysam.AlignedSegment): A pysam AlignedSegment object representing
                                     a mapped read.

    Returns:
        tuple: (variant_pos_in_seq, ref_allele, alt_allele)
               Returns (None, None, None) if no single SNV is found or
               if the MD tag format is not as expected for a simple SNV.
    """
    variant_pos_in_seq = None
    ref_allele = None
    alt_allele = None

    # Check if the read has exactly one mismatch (NM tag)
    if read.has_tag('NM') and read.get_tag('NM') == 1:
        # Check if the read has the MD tag
        if read.has_tag('MD'):
            md_tag = read.get_tag('MD')

            # Regex to parse MD:Z: tag for a single SNV
            # Example: MD:Z:202A67 -> (202, A, 67)
            # This regex captures:
            # Group 1: number of matches before mismatch (e.g., '202')
            # Group 2: reference base at mismatch (e.g., 'A')
            # Group 3: number of matches after mismatch (e.g., '67')
            match = re.match(r'^(\d+)([ACGTN])(\d*)$', md_tag)

            if match:
                # Extract components
                matches_before = int(match.group(1))
                ref_base = match.group(2)

                # The variant position in the query sequence is 0-based,
                # which is simply the number of matches before the mismatch.
                variant_pos_in_seq = matches_before
                ref_allele = ref_base

                # Get the alternative allele from the query sequence at that position
                if read.query_sequence and variant_pos_in_seq < len(read.query_sequence):
                    alt_allele = read.query_sequence[variant_pos_in_seq]
                else:
                    print(f"Warning: Could not get alt allele for read {read.query_name} at pos {variant_pos_in_seq}. Query sequence might be missing or position out of bounds.")
                    alt_allele = "N" # Fallback
            else:
                # MD tag format is not a simple single SNV (e.g., contains indels, multiple mismatches)
                # print(f"Debug: MD tag '{md_tag}' for read {read.query_name} does not match simple SNV pattern.")
                # raise error:
                raise ValueError(f"MD tag '{md_tag}' for read {read.query_name} does not match simple SNV pattern.")
                pass
    # else:
        # print(f"Debug: Read {read.query_name} has NM tag {read.get_tag('NM')} != 1 or no NM tag.")

    return variant_pos_in_seq, ref_allele, alt_allele



def get_genomic_coordinates_bed(dataframe: pd.DataFrame,
                                 sequence_column_name: str,
                                 hg38_fa_path: str,
                                 output_bed_path: str,
                                 id_column_name: str = None,
                                 output_sam_path: str = None,
                                 output_id_map_path: str = None):
    """
    Finds genomic coordinates for sequences in a DataFrame using BWA,
    writes them to a BED file, and optionally saves the SAM alignment file.

    Args:
        dataframe (pd.DataFrame): The input DataFrame containing sequences.
        sequence_column_name (str): The name of the column in the DataFrame
                                    that contains the sequences.
        hg38_fa_path (str): Path to the hg38 reference FASTA file (e.g., 'hg38.fa').
        output_bed_path (str): Path where the output BED file will be written.
        id_column_name (str, optional): The name of the column to use as the
                                        name field in the BED file. If None,
                                        DataFrame index will be used.
        output_sam_path (str, optional): Path where the SAM alignment file
                                         will be saved. If None, a temporary
                                         SAM file is used and cleaned up.
    """

    temp_fasta_path = None
    temp_sam_path_internal = None # Internal variable for temporary SAM file path
    sam_file_for_pysam = None     # Path that pysam will actually read from
    id_mapping = {}               # Dictionary to store short_id -> original_id mapping
    # 1. Create a temporary FASTA file from DataFrame sequences
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.fa') as temp_fasta:
        temp_fasta_path = temp_fasta.name
        for idx, row in dataframe.iterrows():
            # Get the original sequence ID
            original_seq_id = str(row[id_column_name]) if id_column_name else f"df_idx_{idx}"

            # Generate a short, unique ID for the FASTA/SAM/BED files
            # This ensures compatibility with SAM/BAM query name length limits
            short_seq_id = f"seq_{idx}"

            # Store the mapping from short ID to original ID
            id_mapping[short_seq_id] = original_seq_id

            sequence = row[sequence_column_name]
            if not isinstance(sequence, str):
                print(f"Warning: Skipping non-string sequence at index {idx}: {sequence}")
                continue
            temp_fasta.write(f">{short_seq_id}\n{sequence}\n")
    print(f"Temporary FASTA created at: {temp_fasta_path}")
    print(f"Generated {len(id_mapping)} short IDs and stored mapping.")

    # Determine BWA index prefix based on bwa_index_dir
    bwa_ref_prefix = hg38_fa_path # Default to FASTA path

    # 2. Check for BWA index and create if not present
    # BWA index files: .amb, .ann, .bwt, .pac, .sa
    bwa_index_suffixes = [".amb", ".ann", ".bwt", ".pac", ".sa"]
    bwa_index_files = [f"{bwa_ref_prefix}{suffix}" for suffix in bwa_index_suffixes]

    index_exists = all(os.path.exists(f) for f in bwa_index_files)

    if not index_exists:
        print(f"BWA index for {hg38_fa_path} not found at {bwa_ref_prefix}. Creating index...")
        try:
            # Use subprocess to run bwa index command with -p for prefix
            subprocess.run(['bwa', 'index', '-p', bwa_ref_prefix, hg38_fa_path],
                            check=True,
                            capture_output=True,
                            text=True)
            print("BWA index created successfully.")
        except subprocess.CalledProcessError as e:
            print(f"Error creating BWA index: {e.stderr}")
            raise
    else:
        print(f"BWA index for {hg38_fa_path} found at {bwa_ref_prefix}. Skipping indexing.")

    # 3. Run BWA alignment and capture SAM output
    print(f"Running BWA mem on sequences from {temp_fasta_path}...")
    bwa_command = ['bwa', 'mem', bwa_ref_prefix, temp_fasta_path] # Use the prefix for alignment
    try:
        bwa_process = subprocess.run(bwa_command,
                                        capture_output=True,
                                        text=True,
                                        check=True)
        sam_output = bwa_process.stdout
        print("BWA alignment completed.")
    except subprocess.CalledProcessError as e:
        print(f"Error running BWA mem. Stderr: {e.stderr}")
        raise

    # Determine where to write/read the SAM file
    if output_sam_path:
        # User wants to save the SAM file explicitly
        sam_file_for_pysam = output_sam_path
        with open(output_sam_path, 'w') as sam_file:
            sam_file.write(sam_output)
        print(f"SAM alignment saved to: {output_sam_path}")
    else:
        # Use a temporary SAM file
        with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.sam') as temp_sam_file:
            temp_sam_path_internal = temp_sam_file.name
            temp_sam_file.write(sam_output)
        sam_file_for_pysam = temp_sam_path_internal
        print(f"Temporary SAM created at: {temp_sam_path_internal}")

    # 4. Parse SAM output using pysam and prepare BED data
    bed_records = []
    # pysam.AlignmentFile can directly read from the SAM file (either temp or user-specified)
    with pysam.AlignmentFile(sam_file_for_pysam, "r") as samfile:
        for read in samfile.fetch(until_eof=True):
            # Skip unmapped reads and secondary/supplementary alignments
            if read.is_unmapped or read.is_secondary or read.is_supplementary:
                continue

            chrom = samfile.get_reference_name(read.reference_id)
            start = read.reference_start  # 0-based start
            end = read.reference_end      # 0-based exclusive end (matches BED end)

            # Get the short ID from the SAM file (read.query_name)
            short_name_from_sam = read.query_name
            # Look up the original ID using the mapping table
            # Use .get() with a fallback in case a short_name isn't found in mapping (shouldn't happen if logic is correct)
            original_name_for_bed = id_mapping.get(short_name_from_sam, short_name_from_sam)

            score = read.mapping_quality  # Mapping quality as score
            if read.is_reverse:
                raise ValueError("Reverse reads are not supported in this script. Please ensure all reads are forward-stranded.")
            strand = '+' if not read.is_reverse else '-'
            variant_pos, ref_allele, alt_allele = extract_variant_info(read)

            # Append all information to bed_records
            bed_records.append([
                chrom, start, end, original_name_for_bed, score, strand,
                variant_pos if variant_pos is not None else 'NA', # Use '.' for missing info
                ref_allele if ref_allele is not None else 'NA',
                alt_allele if alt_allele is not None else 'NA'
            ])

    # 5. Write BED file
    print(f"Writing BED file to {output_bed_path}...")
    with open(output_bed_path, 'w') as bed_file:
        for record in bed_records:
            bed_file.write('\t'.join(map(str, record)) + '\n')
    print(f"BED file successfully written to {output_bed_path}")

    # 6. Write ID mapping file if requested
    if output_id_map_path:
        id_map_df = pd.DataFrame(list(id_mapping.items()), columns=['short_id', 'original_id'])
        id_map_df.to_csv(output_id_map_path, index=False)
        print(f"ID mapping file saved to: {output_id_map_path}")


# dict of chr number to refseq chromosome number
chrom_2_refseq = {
    "chr1": "NC_000001.11",
    "chr2": "NC_000002.12",
    "chr3": "NC_000003.12",
    "chr4": "NC_000004.12",
    "chr5": "NC_000005.10",
    "chr6": "NC_000006.12",
    "chr7": "NC_000007.14",
    "chr8": "NC_000008.11",
    "chr9": "NC_000009.12",
    "chr10": "NC_000010.11",
    "chr11": "NC_000011.10",
    "chr12": "NC_000012.12",
    "chr13": "NC_000013.11",
    "chr14": "NC_000014.9",
    "chr15": "NC_000015.10",
    "chr16": "NC_000016.10",
    "chr17": "NC_000017.11",
    "chr18": "NC_000018.10",
    "chr19": "NC_000019.10",
    "chr20": "NC_000020.11",
    "chr21": "NC_000021.9",
    "chr22": "NC_000022.11",
    "chrX": "NC_000023.11",
    "chrY": "NC_000024.10"}

# generate SPDI only for alt:
def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) # I think everything is now 0-based
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'


def get_spdi_chengyu_header(row):
    """
    Returns the SPDI identifier for the given variant using start and variant_pos
    Assumption: allele need to be set beforehands
    """
    if (isNaN(row[my_col_ref_base]) or isNaN(row[my_col_alt_base])):
        row[col_SPDI] = 'NA'
        return row
    # identify the variant chrom-pos-ref-alt pattern
    chrom_pos_ref_alt = f'{row[col_chr]}-{row[col_start]+int(row[col_variant_pos])}-{row[my_col_ref_base]}-{row[my_col_alt_base]}'
    if chrom_pos_ref_alt == "NA":
        raise ValueError('Variant pattern could not be found')
    # create the SPDI identifier
    row[col_SPDI] = create_speedy_chromosomes(chrom_pos_ref_alt, seperator='-', indices=[0,1,2,3])
    return row


if __name__ == "__main__":
    metadata_df_path = "/data/cephfs-2/unmirrored/groups/kircher/IGVF_data_submssion_TM/80K/final_design/MPRA_250326.metadata.tsv.gz"

    metadata_df = pd.read_csv(metadata_df_path, sep="\t", low_memory=False)

    # Apply the safe_eval function to the specified columns
    for col in list_columns:
        metadata_df[col] = metadata_df[col].apply(safe_eval)

    chengyu_names_df = metadata_df.loc[metadata_df[col_name].str.contains("C_positive_neuron_CD")].copy()
    chengyu_names_df['RSIDs'] = chengyu_names_df[col_name].apply(extract_rsid)
    chengyu_names_df

    hg38_fa_path = "/data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/hg38/hg38"
    genomic_region_bed_path = "/data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/C_positive_neuron_CD_hg38_coordinates.bed"
    output_sam_path = "/data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/C_positive_neuron_CD_hg38_alignment.sam"
    try:
        get_genomic_coordinates_bed(
            dataframe=chengyu_names_df,
            sequence_column_name=col_sequence,
            hg38_fa_path=hg38_fa_path,
            output_bed_path=genomic_region_bed_path,
            id_column_name=col_name,
            output_sam_path=output_sam_path,
            output_id_map_path="/data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/name_mapping_table.tsv"
        )

    except Exception as e:
        print(f"\nAn error occurred during execution: {e}")


    # read the resulting bed:
    genomic_region_bed = pd.read_csv(genomic_region_bed_path, sep="\t", header=None)
    genomic_region_bed.columns = [col_chr, col_start, col_end, col_name, 'score', col_strand, col_variant_pos, my_col_ref_base, my_col_alt_base]
    genomic_region_bed


    # drop the chr, start, end, strand, variant_pos
    chengyu_names_df_mod = chengyu_names_df.drop(columns=[col_chr, col_start, col_end, col_strand, col_variant_pos, col_SPDI, col_allele, col_info, col_ref, col_category])
    n_chengyu_expected = chengyu_names_df_mod.shape[0]


    # merge the genomic_region_bed with chengyu_names_df on name
    chengyu_names_df_new = pd.merge(chengyu_names_df_mod, genomic_region_bed, on=col_name, how='inner')

    if chengyu_names_df_new.shape[0] != n_chengyu_expected:
        raise ValueError(f"Expected {n_chengyu_expected} rows, but got {chengyu_names_df_new.shape[0]} rows after merging with genomic coordinates.")
    chengyu_names_df_new[col_start] = chengyu_names_df_new[col_start].astype(int)
    chengyu_names_df_new[col_end] = chengyu_names_df_new[col_end].astype(int)
    # chengyu_names_df_new[col_variant_pos] = chengyu_names_df_new[col_variant_pos].astype(int)

    # set category to "element"
    chengyu_names_df_new[col_category] = "element"

    chengyu_names_df_new
    # if variant_pos is NaN
    chengyu_names_df_new.loc[chengyu_names_df_new[col_variant_pos].notna(), col_category] = "variant"

    # set class to element active control
    chengyu_names_df_new.loc[chengyu_names_df_new[col_category] == "element", col_class] = "element active control"
    chengyu_names_df_new.loc[chengyu_names_df_new[col_category] == "variant", col_class] = "variant positive control"

    # set source to "Chengyu Deng"
    chengyu_names_df_new[col_source] = "Chengyu Deng"

    # allele
    chengyu_names_df_new[col_allele] = 'NA'

    # set variant class
    chengyu_names_df_new[col_variant_class] = 'NA'

    # ref
    chengyu_names_df_new[col_ref] = "GRCh38"

    # info
    chengyu_names_df_new[col_info] = ""

    # for alt set alt allele:
    chengyu_names_df_new.loc[chengyu_names_df_new[col_category] == "variant", col_allele] = "alt"
    chengyu_names_df_new.loc[chengyu_names_df_new[col_category] == "variant", col_variant_class] = "SNV"

    chengyu_names_df_new = chengyu_names_df_new.apply(get_spdi_chengyu_header, axis=1)


    # combine corresponding ref and alt:
    # Do I have to every alt a ref? NO => check for which alts you have a ref and combine them
    chengyu_names_df_no_alt = chengyu_names_df.loc[~chengyu_names_df[col_name].str.contains("_alt_")].copy()

    # get their spdis from the rsids (I have a function for this already)
    chengyu_names_df_ref = chengyu_names_df_no_alt.loc[~chengyu_names_df_no_alt[col_name].str.contains("_NA_")].copy()

    # chengyu alt:
    chengyu_names_df_alt = chengyu_names_df.loc[chengyu_names_df[col_name].str.contains("_alt_")].copy()

    # extract the rsids
    rsid_set_alt = set(chengyu_names_df_alt['RSIDs'].dropna().unique())
    rsid_set_ref = set(chengyu_names_df_ref['RSIDs'].dropna().unique())

    rsids_which_have_alt_and_ref = rsid_set_alt.intersection(rsid_set_ref)
    current_spdi = 0
    # function which gets the dataframe and the rsids and for each rsid if the header contains a alt and the rsid id the spdi is computed and set for both the alt and the ref
    def set_the_SPDI_and_variant_pos_and_allele(row, df, rsids):
        """
        For each rsid in the dataframe, if the header contains a alt and the rsid is in the set of rsids,
        the SPDI is computed and set for both the alt and the ref.
        """
        for rsid in rsids:
            if rsid not in row[col_name]:
                continue

            # if '_ref_' get the spdi info from the df and the alt row and add it to the ref row and set the allele to 'ref' and the variant_class to 'SNV'
            if '_ref_' not in row[col_name]:
                continue
            row[col_allele] = 'ref'
            row[col_variant_class] = 'SNV'
            # get the alt row
            rsid_rows = df.loc[df[col_name].str.contains(f'{rsid}')]

            alt_row = rsid_rows.loc[rsid_rows[col_name].str.contains(f'_alt_')]
            if alt_row.empty:
                raise ValueError(f'No alt row found for {rsid} in the dataframe')

            # get the spdi
            current_spdi = alt_row[col_SPDI].to_list()
            if len(current_spdi) == 0:
                raise ValueError(f'SPDI for {rsid} not found in alt row')
            row[col_SPDI] = current_spdi[0]
        return row

    chengyu_names_df_new_with_SPDI = chengyu_names_df_new.apply(lambda row: set_the_SPDI_and_variant_pos_and_allele(row, chengyu_names_df_new, rsids_which_have_alt_and_ref), axis=1)

    # make lists out of columns which need to be lists
    def make_column_arrays(row):
        """
        create arrays for the required columns
        """
        allele = row[col_allele]
        SPDI = row[col_SPDI]
        variant_pos = row[col_variant_pos]
        variant_class = row[col_variant_class]

        if allele == 'ref' or allele == 'alt':
            row[col_allele] = [allele]
        if isinstance(SPDI, str): # only for alt sequences this is true
            if SPDI != "NA":
                row[col_SPDI] = [SPDI]
        if isNaN(variant_pos):
            row[col_variant_pos] = 'NA'
        elif isinstance(variant_pos, float):
            row[col_variant_pos] = [int(variant_pos)]
        elif isinstance(variant_pos, int):
            row[col_variant_pos] = [int(variant_pos)]
        if isinstance(variant_class, str):
            if row[col_variant_class] in ['SNV', 'indel']:
                    row[col_variant_class] = [variant_class]
        if not isinstance(row[col_class], str):
            print(row[col_name])
        return row

    chengyu_names_df_new_with_SPDI_lists = chengyu_names_df_new_with_SPDI.apply(make_column_arrays, axis=1)

    # write to csv
    interesting_columns = [col_name, col_sequence, col_category, col_class, col_source, col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]
    output_path = "/data/cephfs-2/unmirrored/groups/kircher/IGVF_data_submssion_TM/80K/final_design"
    group_name = "C_positive_neuron_CD"
    # Write DataFrame to TSV file
    chengyu_names_df_new_with_SPDI_lists[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
    os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')
    print(f"Metadata for {group_name} written to {output_path}/{group_name}.metadata.tsv.gz")

Temporary FASTA created at: /tmp/tmpmo_85kq4.fa
Generated 96 short IDs and stored mapping.
BWA index for /data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/hg38/hg38 found at /data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/hg38/hg38. Skipping indexing.
Running BWA mem on sequences from /tmp/tmpmo_85kq4.fa...

An error occurred during execution: [Errno 2] No such file or directory: 'bwa'
Metadata for C_positive_neuron_CD written to /data/cephfs-2/unmirrored/groups/kircher/IGVF_data_submssion_TM/80K/final_design/C_positive_neuron_CD.metadata.tsv.gz


In [85]:
chengyu_names_df_new

,name,sequence,category,class,source,variant_class,RSIDs,chr,start,end,score,strand,variant_pos,tmp_ref_base,tmp_alt_base,allele,ref,info
0,C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A...,GATTGTATAAAGAAAATGTGTATACACACACACACACACACACACA...,variant,variant positive control,Chengyu Deng,NA,rs6813360,chr4,155359187,155359457,60,+,NaN,NaN,NaN,alt,GRCh38,
1,C_positive_neuron_CD:p1_rs55985730_T_G_alt_50_...,GGCGCCTGTAGTCCCAGCTACTTGGGAGGCTGAGGCAGGAGAATGG...,variant,variant positive control,Chengyu Deng,NA,rs55985730,chr7,128776855,128777125,60,+,134.0,T,G,alt,GRCh38,
2,C_positive_neuron_CD:p1_rs7115714_G_A_ref_50_A...,TGGTAATTAAAAGCAAAGAGATCTTTTCTATTTGTATGAGCCCTTC...,variant,variant positive control,Chengyu Deng,NA,rs7115714,chr11,120424017,120424287,60,+,NaN,NaN,NaN,alt,GRCh38,
3,C_positive_neuron_CD:p1_rs9975055_T_G_alt_50_G...,CCCTGCTCCCCAGTTCCCACCAGAAACCCCAAGTGGGTGTTCCAGC...,variant,variant positive control,Chengyu Deng,NA,rs9975055,chr21,44929969,44930239,60,+,134.0,T,G,alt,GRCh38,
4,C_positive_neuron_CD:n1_rs2279982_G_A_alt_50::...,TGCGGGCGCTGGCTGGGCGCTGGGGGCCTCGCTGGAGCCCGCTCTC...,variant,variant positive control,Chengyu Deng,NA,rs2279982,chr2,164841902,164842172,60,+,134.0,G,A,alt,GRCh38,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,C_positive_neuron_CD:p2_rs9926320_G_A_ref_25_A...,GTTTGCAGAGTGGGTGTGGTGCCGGAGCACAGGAGCACCTCTTCTG...,variant,variant positive control,Chengyu Deng,NA,rs9926320,chr16,69094082,69094352,60,+,NaN,NaN,NaN,alt,GRCh38,
92,C_positive_neuron_CD:n1_rs7214382_G_C_alt_50::...,GGACATCCCCAGGGACCCCACCAGCCCGGCCCGTAGCCCAGCGGTG...,variant,variant positive control,Chengyu Deng,NA,rs7214382,chr17,33292592,33292862,60,+,134.0,G,C,alt,GRCh38,
93,GC_GABA_Chengyu:GABA|chr13:86882257-86882526|+...,CCTTTCCTTCTTACTAGGTGTACCAGGTGTCATGGCCACCTCCAGA...,variant,variant positive control,Chengyu Deng,NA,None,chr13,86882256,86882526,60,+,NaN,NaN,NaN,alt,GRCh38,
94,C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_...,CTCCCCCGCACAGCGCAGGCTCTCACTGGGAATCTGCCGGGACCGC...,variant,variant positive control,Chengyu Deng,NA,None,chr4,165327321,165327591,60,+,NaN,NaN,NaN,alt,GRCh38,


In [120]:
chengyu_names_df_new

,name,sequence,class,source,variant_class,RSIDs,chr,start,end,score,strand,variant_pos,tmp_ref_base,tmp_alt_base,category,allele,ref,info
0,C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A...,GATTGTATAAAGAAAATGTGTATACACACACACACACACACACACA...,element active control,Chengyu Deng,NA,rs6813360,chr4,155359187,155359457,60,+,NaN,NaN,NaN,element,NA,GRCh38,
1,C_positive_neuron_CD:p1_rs55985730_T_G_alt_50_...,GGCGCCTGTAGTCCCAGCTACTTGGGAGGCTGAGGCAGGAGAATGG...,variant positive control,Chengyu Deng,SNV,rs55985730,chr7,128776855,128777125,60,+,134.0,T,G,variant,alt,GRCh38,
2,C_positive_neuron_CD:p1_rs7115714_G_A_ref_50_A...,TGGTAATTAAAAGCAAAGAGATCTTTTCTATTTGTATGAGCCCTTC...,element active control,Chengyu Deng,NA,rs7115714,chr11,120424017,120424287,60,+,NaN,NaN,NaN,element,NA,GRCh38,
3,C_positive_neuron_CD:p1_rs9975055_T_G_alt_50_G...,CCCTGCTCCCCAGTTCCCACCAGAAACCCCAAGTGGGTGTTCCAGC...,variant positive control,Chengyu Deng,SNV,rs9975055,chr21,44929969,44930239,60,+,134.0,T,G,variant,alt,GRCh38,
4,C_positive_neuron_CD:n1_rs2279982_G_A_alt_50::...,TGCGGGCGCTGGCTGGGCGCTGGGGGCCTCGCTGGAGCCCGCTCTC...,variant positive control,Chengyu Deng,SNV,rs2279982,chr2,164841902,164842172,60,+,134.0,G,A,variant,alt,GRCh38,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,C_positive_neuron_CD:p2_rs9926320_G_A_ref_25_A...,GTTTGCAGAGTGGGTGTGGTGCCGGAGCACAGGAGCACCTCTTCTG...,element active control,Chengyu Deng,NA,rs9926320,chr16,69094082,69094352,60,+,NaN,NaN,NaN,element,NA,GRCh38,
92,C_positive_neuron_CD:n1_rs7214382_G_C_alt_50::...,GGACATCCCCAGGGACCCCACCAGCCCGGCCCGTAGCCCAGCGGTG...,variant positive control,Chengyu Deng,SNV,rs7214382,chr17,33292592,33292862,60,+,134.0,G,C,variant,alt,GRCh38,
93,GC_GABA_Chengyu:GABA|chr13:86882257-86882526|+...,CCTTTCCTTCTTACTAGGTGTACCAGGTGTCATGGCCACCTCCAGA...,element active control,Chengyu Deng,NA,None,chr13,86882256,86882526,60,+,NaN,NaN,NaN,element,NA,GRCh38,
94,C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_...,CTCCCCCGCACAGCGCAGGCTCTCACTGGGAATCTGCCGGGACCGC...,element active control,Chengyu Deng,NA,None,chr4,165327321,165327591,60,+,NaN,NaN,NaN,element,NA,GRCh38,


In [ ]:
import numpy as np
# drop the chr, start, end, strand, variant_pos
chengyu_names_df_mod = chengyu_names_df.drop(columns=[col_chr, col_start, col_end, col_strand, col_variant_pos, col_SPDI, col_allele, col_info, col_ref, col_category])
n_chengyu_expected = chengyu_names_df_mod.shape[0]


# merge the genomic_region_bed with chengyu_names_df on name
chengyu_names_df_new = pd.merge(chengyu_names_df_mod, genomic_region_bed, on=col_name, how='inner')

if chengyu_names_df_new.shape[0] != n_chengyu_expected:
    raise ValueError(f"Expected {n_chengyu_expected} rows, but got {chengyu_names_df_new.shape[0]} rows after merging with genomic coordinates.")
chengyu_names_df_new[col_start] = chengyu_names_df_new[col_start].astype(int)
chengyu_names_df_new[col_end] = chengyu_names_df_new[col_end].astype(int)
# chengyu_names_df_new[col_variant_pos] = chengyu_names_df_new[col_variant_pos].astype(int)

# set category to "element"
chengyu_names_df_new[col_category] = "element"

def isNaN(num):
    return num != num


chengyu_names_df_new
# if variant_pos is NaN
chengyu_names_df_new.loc[chengyu_names_df_new[col_variant_pos].notna(), col_category] = "variant"

# set class to element active control
chengyu_names_df_new.loc[chengyu_names_df_new[col_category] == "element", col_class] = "element active control"
chengyu_names_df_new.loc[chengyu_names_df_new[col_category] == "variant", col_class] = "variant positive control"

# set source to "Chengyu Deng"
chengyu_names_df_new[col_source] = "Chengyu Deng"

# allele
chengyu_names_df_new[col_allele] = 'NA'

# set variant class
chengyu_names_df_new[col_variant_class] = 'NA'

# ref
chengyu_names_df_new[col_ref] = "GRCh38"

# info
chengyu_names_df_new[col_info] = ""

# for alt set alt allele:
chengyu_names_df_new.loc[chengyu_names_df_new[col_category] == "variant", col_allele] = "alt"
chengyu_names_df_new.loc[chengyu_names_df_new[col_category] == "variant", col_variant_class] = "SNV"


# generate SPDI only for alt:
def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) # I think everything is now 0-based
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'


def get_spdi_chengyu_header(row):
    """
    Returns the SPDI identifier for the given variant using start and variant_pos
    Assumption: allele need to be set beforehands
    """
    if (isNaN(row[my_col_ref_base]) or isNaN(row[my_col_alt_base])):
        row[col_SPDI] = 'NA'
        return row
    # identify the variant chrom-pos-ref-alt pattern
    chrom_pos_ref_alt = f'{row[col_chr]}-{row[col_start]+int(row[col_variant_pos])}-{row[my_col_ref_base]}-{row[my_col_alt_base]}'
    if chrom_pos_ref_alt == "NA":
        raise ValueError('Variant pattern could not be found')
    # create the SPDI identifier
    row[col_SPDI] = create_speedy_chromosomes(chrom_pos_ref_alt, seperator='-', indices=[0,1,2,3])
    return row

chengyu_names_df_new = chengyu_names_df_new.apply(get_spdi_chengyu_header, axis=1)

chengyu_names_df_new

# combine corresponding ref and alt:
# Do I have to every alt a ref? NO => check for which alts you have a ref and combine them
chengyu_names_df_no_alt = chengyu_names_df.loc[~chengyu_names_df[col_name].str.contains("_alt_")].copy()

# get their spdis from the rsids (I have a function for this already)
chengyu_names_df_ref = chengyu_names_df_no_alt.loc[~chengyu_names_df_no_alt[col_name].str.contains("_NA_")].copy()

# chengyu alt:
chengyu_names_df_alt = chengyu_names_df.loc[chengyu_names_df[col_name].str.contains("_alt_")].copy()

# extract the rsids
rsid_set_alt = set(chengyu_names_df_alt['RSIDs'].dropna().unique())
rsid_set_ref = set(chengyu_names_df_ref['RSIDs'].dropna().unique())

rsids_which_have_alt_and_ref = rsid_set_alt.intersection(rsid_set_ref)
current_spdi = 0
# function which gets the dataframe and the rsids and for each rsid if the header contains a alt and the rsid id the spdi is computed and set for both the alt and the ref
def set_the_SPDI_and_variant_pos_and_allele(row, df, rsids):
    """
    For each rsid in the dataframe, if the header contains a alt and the rsid is in the set of rsids,
    the SPDI is computed and set for both the alt and the ref.
    """
    for rsid in rsids:
        if rsid not in row[col_name]:
            continue

        # if '_ref_' get the spdi info from the df and the alt row and add it to the ref row and set the allele to 'ref' and the variant_class to 'SNV'
        if '_ref_' not in row[col_name]:
            continue
        row[col_allele] = 'ref'
        row[col_variant_class] = 'SNV'
        # get the alt row
        rsid_rows = df.loc[df[col_name].str.contains(f'{rsid}')]

        alt_row = rsid_rows.loc[rsid_rows[col_name].str.contains(f'_alt_')]
        if alt_row.empty:
            raise ValueError(f'No alt row found for {rsid} in the dataframe')

        # get the spdi
        current_spdi = alt_row[col_SPDI].to_list()
        if len(current_spdi) == 0:
            raise ValueError(f'SPDI for {rsid} not found in alt row')
        row[col_SPDI] = current_spdi[0]
    return row

chengyu_names_df_new_with_SPDI = chengyu_names_df_new.apply(lambda row: set_the_SPDI_and_variant_pos_and_allele(row, chengyu_names_df_new, rsids_which_have_alt_and_ref), axis=1)

# make lists out of columns which need to be lists
def make_column_arrays(row):
    """
    create arrays for the required columns
    """
    allele = row[col_allele]
    SPDI = row[col_SPDI]
    variant_pos = row[col_variant_pos]
    variant_class = row[col_variant_class]

    if allele == 'ref' or allele == 'alt':
        row[col_allele] = [allele]
    if isinstance(SPDI, str): # only for alt sequences this is true
        if SPDI != "NA":
            row[col_SPDI] = [SPDI]
    if isNaN(variant_pos):
        row[col_variant_pos] = 'NA'
    elif isinstance(variant_pos, float):
        row[col_variant_pos] = [int(variant_pos)]
    elif isinstance(variant_pos, int):
        row[col_variant_pos] = [int(variant_pos)]
    if isinstance(variant_class, str):
        if row[col_variant_class] in ['SNV', 'indel']:
                row[col_variant_class] = [variant_class]
    if not isinstance(row[col_class], str):
        print(row[col_name])
    return row

chengyu_names_df_new_with_SPDI_lists = chengyu_names_df_new_with_SPDI.apply(make_column_arrays, axis=1)

# write to csv
interesting_columns = [col_name, col_sequence, col_category, col_class, col_source, col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]
output_path = "/data/cephfs-2/unmirrored/groups/kircher/IGVF_data_submssion_TM/80K/final_design"
group_name = "C_positive_neuron_CD"
# Write DataFrame to TSV file
chengyu_names_df_new_with_SPDI_lists[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')
print(f"Metadata for {group_name} written to {output_path}/{group_name}.metadata.tsv.gz")

Metadata for C_positive_neuron_CD written to /data/cephfs-2/unmirrored/groups/kircher/IGVF_data_submssion_TM/80K/final_design/C_positive_neuron_CD.metadata.tsv.gz


In [122]:
chengyu_names_df_new_with_SPDI_lists

,name,sequence,class,source,variant_class,RSIDs,chr,start,end,score,strand,variant_pos,tmp_ref_base,tmp_alt_base,category,allele,ref,info,SPDI
0,C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A...,GATTGTATAAAGAAAATGTGTATACACACACACACACACACACACA...,element active control,Chengyu Deng,NA,rs6813360,chr4,155359187,155359457,60,+,NA,NaN,NaN,element,NA,GRCh38,,NA
1,C_positive_neuron_CD:p1_rs55985730_T_G_alt_50_...,GGCGCCTGTAGTCCCAGCTACTTGGGAGGCTGAGGCAGGAGAATGG...,variant positive control,Chengyu Deng,[SNV],rs55985730,chr7,128776855,128777125,60,+,[134],T,G,variant,[alt],GRCh38,,[NC_000007.14:128776989:T:G]
2,C_positive_neuron_CD:p1_rs7115714_G_A_ref_50_A...,TGGTAATTAAAAGCAAAGAGATCTTTTCTATTTGTATGAGCCCTTC...,element active control,Chengyu Deng,NA,rs7115714,chr11,120424017,120424287,60,+,NA,NaN,NaN,element,NA,GRCh38,,NA
3,C_positive_neuron_CD:p1_rs9975055_T_G_alt_50_G...,CCCTGCTCCCCAGTTCCCACCAGAAACCCCAAGTGGGTGTTCCAGC...,variant positive control,Chengyu Deng,[SNV],rs9975055,chr21,44929969,44930239,60,+,[134],T,G,variant,[alt],GRCh38,,[NC_000021.9:44930103:T:G]
4,C_positive_neuron_CD:n1_rs2279982_G_A_alt_50::...,TGCGGGCGCTGGCTGGGCGCTGGGGGCCTCGCTGGAGCCCGCTCTC...,variant positive control,Chengyu Deng,[SNV],rs2279982,chr2,164841902,164842172,60,+,[134],G,A,variant,[alt],GRCh38,,[NC_000002.12:164842036:G:A]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,C_positive_neuron_CD:p2_rs9926320_G_A_ref_25_A...,GTTTGCAGAGTGGGTGTGGTGCCGGAGCACAGGAGCACCTCTTCTG...,element active control,Chengyu Deng,NA,rs9926320,chr16,69094082,69094352,60,+,NA,NaN,NaN,element,NA,GRCh38,,NA
92,C_positive_neuron_CD:n1_rs7214382_G_C_alt_50::...,GGACATCCCCAGGGACCCCACCAGCCCGGCCCGTAGCCCAGCGGTG...,variant positive control,Chengyu Deng,[SNV],rs7214382,chr17,33292592,33292862,60,+,[134],G,C,variant,[alt],GRCh38,,[NC_000017.11:33292726:G:C]
93,GC_GABA_Chengyu:GABA|chr13:86882257-86882526|+...,CCTTTCCTTCTTACTAGGTGTACCAGGTGTCATGGCCACCTCCAGA...,element active control,Chengyu Deng,NA,None,chr13,86882256,86882526,60,+,NA,NaN,NaN,element,NA,GRCh38,,NA
94,C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_...,CTCCCCCGCACAGCGCAGGCTCTCACTGGGAATCTGCCGGGACCGC...,element active control,Chengyu Deng,NA,None,chr4,165327321,165327591,60,+,NA,NaN,NaN,element,NA,GRCh38,,NA


In [112]:
rsids_which_have_alt_and_ref

{'rs275835',
 'rs34241773',
 'rs606742',
 'rs62086577',
 'rs7115714',
 'rs76990668',
 'rs9931091'}

In [ ]:
chengyu_names_df_new

In [100]:
chengyu_names_df_new_with_SPDI

,name,sequence,category,class,source,variant_class,RSIDs,chr,start,end,score,strand,variant_pos,tmp_ref_base,tmp_alt_base,allele,ref,info,SPDI
0,C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A...,GATTGTATAAAGAAAATGTGTATACACACACACACACACACACACA...,variant,variant positive control,Chengyu Deng,NA,rs6813360,chr4,155359187,155359457,60,+,NaN,NaN,NaN,alt,GRCh38,,NA
1,C_positive_neuron_CD:p1_rs55985730_T_G_alt_50_...,GGCGCCTGTAGTCCCAGCTACTTGGGAGGCTGAGGCAGGAGAATGG...,variant,variant positive control,Chengyu Deng,NA,rs55985730,chr7,128776855,128777125,60,+,134.0,T,G,alt,GRCh38,,NC_000007.14:128776989:T:G
2,C_positive_neuron_CD:p1_rs7115714_G_A_ref_50_A...,TGGTAATTAAAAGCAAAGAGATCTTTTCTATTTGTATGAGCCCTTC...,variant,variant positive control,Chengyu Deng,NA,rs7115714,chr11,120424017,120424287,60,+,NaN,NaN,NaN,alt,GRCh38,,NA
3,C_positive_neuron_CD:p1_rs9975055_T_G_alt_50_G...,CCCTGCTCCCCAGTTCCCACCAGAAACCCCAAGTGGGTGTTCCAGC...,variant,variant positive control,Chengyu Deng,NA,rs9975055,chr21,44929969,44930239,60,+,134.0,T,G,alt,GRCh38,,NC_000021.9:44930103:T:G
4,C_positive_neuron_CD:n1_rs2279982_G_A_alt_50::...,TGCGGGCGCTGGCTGGGCGCTGGGGGCCTCGCTGGAGCCCGCTCTC...,variant,variant positive control,Chengyu Deng,NA,rs2279982,chr2,164841902,164842172,60,+,134.0,G,A,alt,GRCh38,,NC_000002.12:164842036:G:A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,C_positive_neuron_CD:p2_rs9926320_G_A_ref_25_A...,GTTTGCAGAGTGGGTGTGGTGCCGGAGCACAGGAGCACCTCTTCTG...,variant,variant positive control,Chengyu Deng,NA,rs9926320,chr16,69094082,69094352,60,+,NaN,NaN,NaN,alt,GRCh38,,NA
92,C_positive_neuron_CD:n1_rs7214382_G_C_alt_50::...,GGACATCCCCAGGGACCCCACCAGCCCGGCCCGTAGCCCAGCGGTG...,variant,variant positive control,Chengyu Deng,NA,rs7214382,chr17,33292592,33292862,60,+,134.0,G,C,alt,GRCh38,,NC_000017.11:33292726:G:C
93,GC_GABA_Chengyu:GABA|chr13:86882257-86882526|+...,CCTTTCCTTCTTACTAGGTGTACCAGGTGTCATGGCCACCTCCAGA...,variant,variant positive control,Chengyu Deng,NA,None,chr13,86882256,86882526,60,+,NaN,NaN,NaN,alt,GRCh38,,NA
94,C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_...,CTCCCCCGCACAGCGCAGGCTCTCACTGGGAATCTGCCGGGACCGC...,variant,variant positive control,Chengyu Deng,NA,None,chr4,165327321,165327591,60,+,NaN,NaN,NaN,alt,GRCh38,,NA


In [72]:
chengyu_names_df_new_with_SPDI

,name,sequence,category,class,source,ref,variant_class,SPDI,allele,info,RSIDs,chr,start,end,score,strand,variant_pos,tmp_ref_base,tmp_alt_base
0,C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A...,GATTGTATAAAGAAAATGTGTATACACACACACACACACACACACA...,element,element active control,Chengyu Deng,GRCh38,[SNV],NA,NA,NaN,rs6813360,chr4,155359187,155359457,60,+,.,.,.
1,C_positive_neuron_CD:p1_rs55985730_T_G_alt_50_...,GGCGCCTGTAGTCCCAGCTACTTGGGAGGCTGAGGCAGGAGAATGG...,variant,variant positive control,Chengyu Deng,GRCh38,[SNV],NC_000007.14:128776989:T:G,alt,NaN,rs55985730,chr7,128776855,128777125,60,+,134,T,G
2,C_positive_neuron_CD:p1_rs7115714_G_A_ref_50_A...,TGGTAATTAAAAGCAAAGAGATCTTTTCTATTTGTATGAGCCCTTC...,element,element active control,Chengyu Deng,GRCh38,[SNV],NA,NA,NaN,rs7115714,chr11,120424017,120424287,60,+,.,.,.
3,C_positive_neuron_CD:p1_rs9975055_T_G_alt_50_G...,CCCTGCTCCCCAGTTCCCACCAGAAACCCCAAGTGGGTGTTCCAGC...,variant,variant positive control,Chengyu Deng,GRCh38,[SNV],NC_000021.9:44930103:T:G,alt,NaN,rs9975055,chr21,44929969,44930239,60,+,134,T,G
4,C_positive_neuron_CD:n1_rs2279982_G_A_alt_50::...,TGCGGGCGCTGGCTGGGCGCTGGGGGCCTCGCTGGAGCCCGCTCTC...,variant,variant positive control,Chengyu Deng,GRCh38,[SNV],NC_000002.12:164842036:G:A,alt,NaN,rs2279982,chr2,164841902,164842172,60,+,134,G,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,C_positive_neuron_CD:p2_rs9926320_G_A_ref_25_A...,GTTTGCAGAGTGGGTGTGGTGCCGGAGCACAGGAGCACCTCTTCTG...,element,element active control,Chengyu Deng,GRCh38,[SNV],NA,NA,NaN,rs9926320,chr16,69094082,69094352,60,+,.,.,.
92,C_positive_neuron_CD:n1_rs7214382_G_C_alt_50::...,GGACATCCCCAGGGACCCCACCAGCCCGGCCCGTAGCCCAGCGGTG...,variant,variant positive control,Chengyu Deng,GRCh38,[SNV],NC_000017.11:33292726:G:C,alt,NaN,rs7214382,chr17,33292592,33292862,60,+,134,G,C
93,GC_GABA_Chengyu:GABA|chr13:86882257-86882526|+...,CCTTTCCTTCTTACTAGGTGTACCAGGTGTCATGGCCACCTCCAGA...,element,element active control,Chengyu Deng,GRCh38,None,NA,NA,NaN,None,chr13,86882256,86882526,60,+,.,.,.
94,C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_...,CTCCCCCGCACAGCGCAGGCTCTCACTGGGAATCTGCCGGGACCGC...,element,element active control,Chengyu Deng,GRCh38,None,NA,NA,NaN,None,chr4,165327321,165327591,60,+,.,.,.


In [64]:
current_spdi

0

In [55]:
rsids_which_have_alt_and_ref

{'rs275835',
 'rs34241773',
 'rs606742',
 'rs62086577',
 'rs7115714',
 'rs76990668',
 'rs9931091'}

In [108]:
set(chengyu_names_df_ref['RSIDs'].dropna().unique()) - rsids_which_have_alt_and_ref

{'rs1005658',
 'rs10061048',
 'rs10079318',
 'rs10881994',
 'rs11086102',
 'rs11164122',
 'rs11170351',
 'rs112548538',
 'rs114772924',
 'rs11543742',
 'rs11616803',
 'rs11757302',
 'rs117901939',
 'rs11876',
 'rs12631337',
 'rs12773142',
 'rs12944649',
 'rs171632',
 'rs17628',
 'rs183841818',
 'rs2008701',
 'rs2278405',
 'rs2293578',
 'rs2305800',
 'rs2776306',
 'rs2999546',
 'rs353548',
 'rs35714',
 'rs3803686',
 'rs4333644',
 'rs456520',
 'rs4982404',
 'rs6062301',
 'rs6119279',
 'rs6457750',
 'rs66500423',
 'rs6791336',
 'rs6813360',
 'rs6916842',
 'rs6919110',
 'rs72692803',
 'rs77845395',
 'rs8049948',
 'rs817339',
 'rs910033',
 'rs9926320'}

In [ ]:
chengyu_names_df_new_with_SPDI

In [ ]:
hg38_fa_path = "/data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/hg38"
output_bed_file = "/data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/projects/metadata/C_positive_neuron_CD/C_positive_neuron_CD_hg38_coordinates.bed"

In [ ]:
rsid_set_alt

{'rs10971392',
 'rs115202710',
 'rs1159310',
 'rs11913731',
 'rs12411493',
 'rs154568',
 'rs17175954',
 'rs1763596',
 'rs17653',
 'rs2224873',
 'rs2279982',
 'rs2529627',
 'rs275835',
 'rs28365157',
 'rs34241773',
 'rs34761481',
 'rs35341445',
 'rs55985730',
 'rs5771096',
 'rs584415',
 'rs60027384',
 'rs606742',
 'rs61828645',
 'rs61901488',
 'rs62028868',
 'rs62086577',
 'rs6798807',
 'rs7072579',
 'rs7107603',
 'rs7115714',
 'rs7156717',
 'rs7214382',
 'rs7582466',
 'rs76990668',
 'rs7794324',
 'rs923347',
 'rs9931091',
 'rs9975055'}

In [ ]:
rsid_set_ref

{'rs1005658',
 'rs10061048',
 'rs10079318',
 'rs10881994',
 'rs11086102',
 'rs11164122',
 'rs11170351',
 'rs112548538',
 'rs114772924',
 'rs11543742',
 'rs11616803',
 'rs11757302',
 'rs117901939',
 'rs11876',
 'rs12631337',
 'rs12773142',
 'rs12944649',
 'rs171632',
 'rs17628',
 'rs183841818',
 'rs2008701',
 'rs2278405',
 'rs2293578',
 'rs2305800',
 'rs275835',
 'rs2776306',
 'rs2999546',
 'rs34241773',
 'rs353548',
 'rs35714',
 'rs3803686',
 'rs4333644',
 'rs456520',
 'rs4982404',
 'rs6062301',
 'rs606742',
 'rs6119279',
 'rs62086577',
 'rs6457750',
 'rs66500423',
 'rs6791336',
 'rs6813360',
 'rs6916842',
 'rs6919110',
 'rs7115714',
 'rs72692803',
 'rs76990668',
 'rs77845395',
 'rs8049948',
 'rs817339',
 'rs910033',
 'rs9926320',
 'rs9931091'}

In [ ]:
# do the 5 sequences with NA:
chengyu_names_df_NA = chengyu_names_df_no_alt.loc[chengyu_names_df_no_alt[col_name].str.contains("_NA_")].copy()
chengyu_names_df_NA

In [ ]:
chengyu_names_df_ref

In [ ]:
import requests

rsid = "rs6813360"
url = f"https://rest.ensembl.org/variation/human/{rsid}?content-type=application/json"

response = requests.get(url, headers={"Content-Type": "application/json"})

if response.status_code == 200:
    data = response.json()
    for mapping in data.get("mappings", []):
        print(f"Chromosome: {mapping['seq_region_name']}, "
              f"Start: {mapping['start']}, End: {mapping['end']}, "
              f"Strand: {mapping['strand']}",
              f"Reference: {mapping['assembly_name']}")
        if mapping['assembly_name'] != "GRCh38":
            print("Warning: assembly is different than expected")

else:
    print(f"Error: {response.status_code}")


Chromosome: 4, Start: 155359322, End: 155359322, Strand: 1 Reference: GRCh38


cases I tested